# Legal AI Public Discourse Analyzer

**HCDE 530 — Mini Project 2 (Research Track)**

> This tool does not automate grounded theory. It supports early-stage qualitative analysis by surfacing recurring themes, frequency signals, illustrative excerpts, and short researcher memos. The human researcher still interprets and validates the findings.

## Research question

What recurring themes appear in **Reddit-only** public discourse about legal AI tools, and which excerpts best illustrate each theme?

## Pipeline (Reddit-only scope)

1. **Collect** — live Reddit API (optional) or curated sample ingestion
2. **Clean** — normalize text, deduplicate short rows
3. **Code themes** — keyword coding with TF-IDF clustering fallback
4. **Summarize** — frequency-ranked themes + illustrative excerpts
5. **Memo** — short researcher notes for early interpretation
6. **Visualize** — one bar chart of top themes

**Out of scope for this showcase:** LinkedIn scraping, blog scraping, deposition prep, Teachable Machine, production frontend.

In [ ]:
from pathlib import Path
import sys

# Notebook kernel cwd should be MiniProject2/
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from IPython.display import Image, Markdown, display

from src.pipeline import POSITIONING, run_pipeline

display(Markdown(POSITIONING))

## 1. Run pipeline

Set `use_live_reddit=True` only when `MiniProject2/.env` contains Reddit API credentials. Default uses the curated sample corpus so the notebook runs offline.

In [ ]:
artifacts = run_pipeline(use_api=False)

raw = pd.read_csv(artifacts["raw"])
processed = pd.read_csv(artifacts["processed"])
themes = pd.read_csv(artifacts["theme_summary"])
excerpts = pd.read_csv(artifacts["excerpts"])

print(f"Raw Reddit rows: {len(raw)}")
print(f"Processed corpus: {len(processed)}")
print(f"Themes identified: {themes['theme'].nunique()}")

## 2. Frequency-ranked themes

These counts prioritize where to look first. They do **not** replace researcher judgment about meaning or saturation.

In [ ]:
themes[["theme_label", "count", "percentage", "mean_reddit_score", "top_subreddits"]]

## 3. Illustrative excerpts (top engagement per theme)

In [ ]:
excerpts.sort_values(["theme", "rank"])[
    ["theme_label", "rank", "subreddit", "score", "excerpt", "source_url"]
].head(12)

## 4. Researcher memos and chart

In [ ]:
memo_text = artifacts["memos"].read_text(encoding="utf-8")
preview = memo_text if len(memo_text) <= 4000 else memo_text[:4000] + "\n\n..."
display(Markdown(preview))
display(Image(filename=str(artifacts["chart"])))